# Medical Image AI Suite
### دفتر تفاعلي لتجهيز وتدريب نماذج الذكاء الاصطناعي على الصور الطبية

---

## المحاور الأربعة:
1. **توحيد وتجهيز البيانات** (Preprocessing) - DICOM/JPG → NumPy
2. **تعلّم شبه خاضع للإشراف** (Semi-supervised) - استخراج إشارات ضعيفة من التقارير
3. **توليد بيانات اصطناعية** (Synthetic Data) - MedGAN
4. **توليد تقارير تلقائية** (Report Generation) - VLM

---

In [ ]:
# ============================================================
# الخلية 1: التثبيت والإعداد
# ============================================================

import sys
import os
import json
import time
from pathlib import Path

print("Python:", sys.version)

# تثبيت التبعيات الأساسية
!pip install -q pydicom SimpleITK opencv-python-headless Pillow scikit-image numpy scipy pyyaml tqdm 2>/dev/null

# تثبيت PyTorch
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu 2>/dev/null

# تثبيت مكتبات NLP (اختياري)
!pip install -q transformers datasets tokenizers accelerate sentencepiece 2>/dev/null

# تثبيت أدوات التقييم (اختياري)
!pip install -q matplotlib seaborn pandas tqdm 2>/dev/null

print("✓ تم تثبيت التبعيات الأساسية")

In [ ]:
# ============================================================
# الخلية 2: تحميل المشروع وتهيئة البيئة
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# دعم العربية في Matplotlib
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC[wght].ttf')
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# إنشاء هيكل المجلدات
BASE_DIR = Path("/content/medical-ai-suite")
for subdir in ["data/raw", "data/processed", "data/labels", "data/synthetic", "data/reports", "models", "logs"]:
    (BASE_DIR / subdir).mkdir(parents=True, exist_ok=True)

print(f"المجلد الأساسي: {BASE_DIR}")
print(f"✓ تم إنشاء هيكل المجلدات")

---
## المحور 1: توحيد وتجهيز البيانات (Preprocessing)
---

In [ ]:
# ============================================================
# الخلية 3: معالجة ملفات DICOM
# ============================================================

import pydicom
import cv2
from PIL import Image
from scipy.ndimage import gaussian_filter
from tqdm.notebook import tqdm

# ===== نطاقات Windowing الطبية =====
PRESET_WINDOWS = {
    "lung":         {"center": -600, "width": 1500, "desc": "نافذة الرئة"},
    "mediastinum":  {"center": 40,   "width": 400,  "desc": "نافذة المنصف"},
    "bone":         {"center": 400,  "width": 1800, "desc": "نافذة العظام"},
    "brain":        {"center": 40,   "width": 80,   "desc": "نافذة الدماغ"},
    "liver":        {"center": 60,   "width": 150,  "desc": "نافذة الكبد"},
}


def apply_windowing(pixel_array, window_name="lung"):
    """تطبيق Windowing طبي"""
    w = PRESET_WINDOWS[window_name]
    lower = w["center"] - w["width"] / 2
    upper = w["center"] + w["width"] / 2
    windowed = np.clip(pixel_array, lower, upper)
    windowed = ((windowed - lower) / w["width"]) * 255.0
    return np.clip(windowed, 0, 255).astype(np.uint8)


def load_dicom(filepath):
    """تحميل ملف DICOM"""
    ds = pydicom.dcmread(filepath, force=True)
    pixel_array = ds.pixel_array.astype(np.float64)
    if hasattr(ds, 'RescaleSlope') and ds.RescaleSlope:
        pixel_array = pixel_array * float(ds.RescaleSlope)
    if hasattr(ds, 'RescaleIntercept') and ds.RescaleIntercept:
        pixel_array = pixel_array + float(ds.RescaleIntercept)
    return pixel_array, ds


def preprocess_dicom(filepath, target_size=(512, 512), window="lung"):
    """معالجة ملف DICOM كامل"""
    pixel_array, ds = load_dicom(filepath)
    windowed = apply_windowing(pixel_array, window)
    # تغيير الحجم
    img = Image.fromarray(windowed)
    img = img.resize(target_size, Image.BILINEAR)
    return np.array(img), ds


# === تجربة مع بيانات تجريبية ===
print("إنشاء بيانات تجريبية للعرض...")

# إنشاء صورة تجريبية تحاكي صورة صدر
def create_sample_xray(size=512):
    """إنشاء صورة أشعة سينية تجريبية"""
    img = np.zeros((size, size), dtype=np.float64)
    # خلفية داكنة
    img[:, :] = 20
    # الرئة اليمنى (بيضاء فاتحة)
    yy, xx = np.ogrid[:size, :size]
    right_lung = ((xx - size*0.3)**2 / (size*0.22)**2 + (yy - size*0.45)**2 / (size*0.35)**2) < 1
    img[right_lung] = 180
    # الرئة اليسرى
    left_lung = ((xx - size*0.7)**2 / (size*0.2)**2 + (yy - size*0.45)**2 / (size*0.32)**2) < 1
    img[left_lung] = 170
    # القلب (أبيض)
    heart = ((xx - size*0.5)**2 / (size*0.12)**2 + (yy - size*0.48)**2 / (size*0.15)**2) < 1
    img[heart] = 220
    # عمود فقري
    spine = (np.abs(xx - size*0.5) < size*0.03) & (yy > size*0.2) & (yy < size*0.8)
    img[spine] = 200
    # ضوضاء
    img += np.random.normal(0, 5, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8)

# إنشاء وتصور صورة تجريبية
sample_img = create_sample_xray()

# تطبيق نوافذ مختلفة
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(sample_img, cmap='gray')
axes[0].set_title('Original (Raw HU)', fontsize=12)

for i, win_name in enumerate(["lung", "mediastinum", "bone"]):
    windowed = apply_windowing(sample_img.astype(np.float64), win_name)
    axes[i+1].imshow(windowed, cmap='gray')
    axes[i+1].set_title(f'{PRESET_WINDOWS[win_name]["desc"]}', fontsize=12)

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'windowing_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ تم عرض نوافذ Windowing مختلفة")

---
## المحور 2: استخراج الكيانات الطبية (NER) + إشارات ضعيفة
---

In [ ]:
# ============================================================
# الخلية 4: معالجة النصوص واستخراج الكيانات
# ============================================================

import re
import unicodedata

# ===== عينة تقارير طبية عربية ====
SAMPLE_REPORTS = [
    {
        "id": 1,
        "text": """تقرير الفحص الشعاعي للصدر
القصة المرضية: مريض عمره 45 سنة يشكو من سعال وألم صدري منذ أسبوع

النتائج:
يوجد التهاب رئوي في الفص السفلي الأيمن مع ارتشاح رئوي
انصباب جنبي بسيط في الجانب الأيسر
القلب بحجم طبيعي
عمود فقري طبيعي

الاستنتاج:
التهاب رئوي أيمن سفلية مع انصباب جنبي أيسر بسيط
يُنصح بالعلاج بالمضادات الحيوية والمتابعة""",
    },
    {
        "id": 2,
        "text": """تقرير الأشعة السينية للصدر
الشكوى: ضيق تنفس حاد

النتائج:
لا يوجد ارتشاح رئوي
لا يوجد انصباب جنبي
تضخم القلب مع توسع في المنصف
الرئتان سليمتان

الاستنتاج:
تضخم قلبي مع وسطاني متمدد
يُنصح بإيكو القلب""",
    },
    {
        "id": 3,
        "text": """تقرير صورة الصدر
القصة: رضح صدري بعد حادث سير

النتائج:
كسر في الضلع السادس والسابع الأيمن
ورم دموي في الرئة اليمنى
استرواح صدر يساري بسيط
انصباب جنبي يساري معتدل

الاستنتاج:
كسور ضلعية متعددة مع استرواح صدر ورذحة رئوية
يُنصح بإدخال أنبوب صدري""",
    },
    {
        "id": 4,
        "text": """تقرير CT Scan للصدر
يوجد ورم في الفص العلوي الأيمن حجمه 3.5 سم
نقائل في الغدد اللمفاوية المنصفية
تسلخ أورطي """,
    },
    {
        "id": 5,
        "text": """صورة صدر طبيعية
الرئتان سليمتان بدون ارتشاح أو انصباب
القلب والمنصف ضمن الحدود الطبيعية
لا يوجد أي نتائج مرضية""",
    },
]

print(f"تم تحميل {len(SAMPLE_REPORTS)} تقرير طبي عينة")

# ===== تنظيف النصوص =====
ARABIC_DIACRITICS = re.compile(
    r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]"
)

def clean_arabic_text(text):
    """تنظيف النص العربي الطبي"""
    text = ARABIC_DIACRITICS.sub("", text)
    text = text.replace("إ", "ا").replace("أ", "ا").replace("آ", "ا")
    text = text.replace("ة", "ه")
    text = re.sub(r"\s+", " ", text).strip()
    return text

for report in SAMPLE_REPORTS:
    report["cleaned"] = clean_arabic_text(report["text"])

print(f"\n--- مثال على تقرير منظم ---")
print(SAMPLE_REPORTS[0]["cleaned"][:300])

In [ ]:
# ============================================================
# الخلية 5: استخراج الكيانات الطبية (NER) + إشارات ضعيفة
# ============================================================

# ===== قاموس طبي مصغر =====
MEDICAL_DICTIONARY = {
    "pneumonia":              {"ar": "التهاب رئوي", "cat": "DISEASE"},
    "infiltration":           {"ar": "ارتشاح رئوي", "cat": "DISEASE"},
    "pleural_effusion":       {"ar": "انصباب جنبي", "cat": "DISEASE"},
    "pneumothorax":           {"ar": "استرواح صدر", "cat": "DISEASE"},
    "cardiomegaly":           {"ar": "تضخم القلب", "cat": "DISEASE"},
    "fracture":               {"ar": "كسر", "cat": "FINDING"},
    "hematoma":               {"ar": "ورم دموي", "cat": "FINDING"},
    "tumor":                  {"ar": "ورم", "cat": "FINDING"},
    "metastasis":             {"ar": "نقائل", "cat": "FINDING"},
    "aortic_dissection":      {"ar": "تسلخ أورطي", "cat": "DISEASE"},
    "normal":                 {"ar": "طبيعي", "cat": "NORMAL"},
    "right":                  {"ar": "ايمن", "cat": "LATERALITY"},
    "left":                   {"ar": "ايسر", "cat": "LATERALITY"},
    "bilateral":              {"ar": "ثنائي الجانب", "cat": "LATERALITY"},
}

NEGATION_WORDS = [
    "لا يوجد", "لا يوجد", "بدون", "غير موجود", "سالب", "طبيعي",
    "سليم", "within normal", "no evidence", "negative", "no", "without",
]


def extract_entities(text):
    """استخراج الكيانات من النص"""
    entities = []
    text_lower = text.replace("إ", "ا").replace("أ", "ا").replace("ة", "ه")

    for eng_name, info in MEDICAL_DICTIONARY.items():
        ar_term = info["ar"]
        if ar_term in text_lower or eng_name.lower() in text_lower:
            # كشف النفي
            idx = text_lower.find(ar_term)
            context = text_lower[max(0, idx-40):min(len(text_lower), idx+40)]
            is_negated = any(neg in context for neg in NEGATION_WORDS)

            entities.append({
                "text": ar_term,
                "english": eng_name,
                "category": info["cat"],
                "is_negated": is_negated,
                "confidence": 0.3 if is_negated else 1.0,
            })

    return entities


def extract_weak_labels(text):
    """استخراج إشارات ضعيفة (ثنائية) من التقرير"""
    entities = extract_entities(text)
    labels = {}

    for entity in entities:
        if entity["category"] in ("DISEASE", "FINDING"):
            key = entity["english"]
            if entity["is_negated"]:
                labels[key] = 0.0
            else:
                labels[key] = 1.0

    # إشارة عامة
    diseases = [e for e in entities if e["category"] in ("DISEASE", "FINDING") and not e["is_negated"]]
    labels["abnormal"] = 1.0 if diseases else 0.0

    return labels


# === استخراج الكيانات من جميع التقارير ===
all_entities = []
all_labels = []

for report in SAMPLE_REPORTS:
    entities = extract_entities(report["cleaned"])
    labels = extract_weak_labels(report["cleaned"])
    all_entities.append(entities)
    all_labels.append(labels)

    print(f"\n--- تقرير {report['id']} ---")
    for e in entities:
        status = "[منفي]" if e["is_negated"] else "[موجود]"
        print(f"  {status} {e['text']} ({e['english']}) - {e['category']}")
    print(f"  الإشارات: {labels}")

In [ ]:
# ============================================================
# الخلية 6: بناء مصفوفة الإشارات الضعيفة + تصور
# ============================================================

import pandas as pd
import seaborn as sns

# بناء DataFrame
all_keys = sorted(set(k for labels in all_labels for k in labels))
label_matrix = np.zeros((len(all_labels), len(all_keys)), dtype=np.float32)

for i, labels in enumerate(all_labels):
    for j, key in enumerate(all_keys):
        label_matrix[i, j] = labels.get(key, 0.0)

df_labels = pd.DataFrame(label_matrix, columns=all_keys, index=[f"Report {i+1}" for i in range(len(all_labels))])

print("مصفوفة الإشارات الضعيفة (Weak Labels):")
print(df_labels.T.to_string())

# تصور المصفوفة
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(df_labels, annot=True, cmap="RdYlGn_r", center=0.5, ax=ax, fmt=".1f", cbar_kws={"label": "Label"})
ax.set_title("Weak Label Matrix - Medical Reports", fontsize=14)
ax.set_xlabel("Reports", fontsize=12)
ax.set_ylabel("Conditions", fontsize=12)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'weak_labels_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# حفظ المصفوفة
np.save(BASE_DIR / 'data' / 'labels' / 'weak_labels.npy', label_matrix)
with open(BASE_DIR / 'data' / 'labels' / 'class_names.json', 'w', encoding='utf-8') as f:
    json.dump(all_keys, f, ensure_ascii=False, indent=2)

print(f"\n✓ تم حفظ المصفوفة: {label_matrix.shape}")
print(f"✓ عدد الفئات: {len(all_keys)}")
print(f"✓ كثافة التسميات: {np.mean(label_matrix != 0):.2%}")

---
## المحور 3: تعزيز البيانات + توليد اصطناعي
---

In [ ]:
# ============================================================
# الخلية 7: تعزيز البيانات (Data Augmentation)
# ============================================================

from scipy.ndimage import rotate as scipy_rotate, gaussian_filter

def augment_image(img, seed=None):
    """تعزيز صورة طبية واحدة"""
    if seed is not None:
        np.random.seed(seed)

    augmented = img.astype(np.float32)

    # دوران عشوائي
    angle = np.random.uniform(-15, 15)
    augmented = scipy_rotate(augmented, angle, reshape=False, mode='reflect')

    # ضوضاء غاوسية
    if np.random.random() > 0.5:
        augmented += np.random.normal(0, 3, augmented.shape)

    # تعديل السطوع
    brightness = np.random.uniform(0.8, 1.2)
    augmented = np.clip(augmented * brightness, 0, 255)

    # قلب أفقي
    if np.random.random() > 0.5:
        augmented = np.fliplr(augmented)

    return augmented.astype(np.uint8)


# توليد مجموعة صور تجريبية
print("توليد صور تجريبية وتعزيزها...")
original_images = [create_sample_xray() for _ in range(10)]

# تعزيز كل صورة
augmented_per_image = 5
all_images = original_images.copy()

for i, img in enumerate(original_images):
    for j in range(augmented_per_image):
        aug = augment_image(img, seed=i*100+j)
        all_images.append(aug)

print(f"الأصلية: {len(original_images)} → بعد التعزيز: {len(all_images)}")

# تصور بعض الأمثلة
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i in range(6):
    axes[0, i].imshow(original_images[i], cmap='gray')
    axes[0, i].set_title(f'Original {i+1}')
    axes[0, i].axis('off')

    axes[1, i].imshow(all_images[10 + i], cmap='gray')
    axes[1, i].set_title(f'Augmented {i+1}')
    axes[1, i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'augmentation_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ تم تعزيز البيانات بنجاح")

In [ ]:
# ============================================================
# الخلية 8: توليد صور اصطناعية بسيط (Demo GAN)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"الجهاز: {device}")

IMG_SIZE = 64
LATENT_DIM = 128

# تحويل الصور إلى الحجم المطلوب
def prepare_images_for_gan(images, size=IMG_SIZE):
    prepared = []
    for img in images:
        pil = Image.fromarray(img)
        pil = pil.resize((size, size), Image.BILINEAR)
        arr = np.array(pil).astype(np.float32) / 127.5 - 1  # [-1, 1]
        prepared.append(arr)
    return np.array(prepared)


# بناء Generator بسيط
class SimpleGenerator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 256 * 4 * 4),
            nn.BatchNorm1d(256 * 4 * 4),
            nn.ReLU(True),
        )
        self.deconv = nn.Sequential(
            nn.Upsample(scale_factor=2), nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Upsample(scale_factor=2), nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Upsample(scale_factor=2), nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Upsample(scale_factor=2), nn.Conv2d(32, 1, 3, padding=1), nn.Tanh(),
        )

    def forward(self, z):
        x = self.fc(z).view(z.size(0), 256, 4, 4)
        return self.deconv(x)


class SimpleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.LeakyReLU(0.2), nn.Dropout2d(0.25),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.LeakyReLU(0.2), nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.LeakyReLU(0.2), nn.Dropout2d(0.25),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.LeakyReLU(0.2), nn.Dropout2d(0.25),
        )
        self.fc = nn.Sequential(
            nn.Linear(256 * 4 * 4, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        x = self.model(img)
        return self.fc(x.view(x.size(0), -1))


# تجهيز البيانات
real_imgs = prepare_images_for_gan(original_images)
X = torch.FloatTensor(real_imgs).unsqueeze(1).to(device)

print(f"بيانات التدريب: {X.shape}")

# بناء النماذج
G = SimpleGenerator(LATENT_DIM).to(device)
D = SimpleDiscriminator().to(device)

g_opt = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = optim.Adam(D.parameters(), lr=1e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()

dataloader = DataLoader(TensorDataset(X), batch_size=8, shuffle=True, drop_last=True)

# تدريب
EPOCHS = 100
g_losses, d_losses = [], []

print(f"\nبدء تدريب GAN: {EPOCHS} حقب")
for epoch in range(EPOCHS):
    G.train(); D.train()
    epoch_g, epoch_d = 0, 0

    for (batch,) in dataloader:
        bs = batch.size(0)
        real_labels = torch.ones(bs, 1).to(device)
        fake_labels = torch.zeros(bs, 1).to(device)

        # Discriminator
        d_opt.zero_grad()
        d_real = D(batch)
        d_loss_real = criterion(d_real, real_labels)

        z = torch.randn(bs, LATENT_DIM).to(device)
        fake = G(z)
        d_fake = D(fake.detach())
        d_loss_fake = criterion(d_fake, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        d_opt.step()

        # Generator
        g_opt.zero_grad()
        z = torch.randn(bs, LATENT_DIM).to(device)
        fake = G(z)
        g_loss = criterion(D(fake), real_labels)
        g_loss.backward()
        g_opt.step()

        epoch_g += g_loss.item()
        epoch_d += d_loss.item()

    g_losses.append(epoch_g / len(dataloader))
    d_losses.append(epoch_d / len(dataloader))

    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | G_loss={g_losses[-1]:.4f} | D_loss={d_losses[-1]:.4f}")

print("✓ انتهى تدريب GAN")

In [ ]:
# ============================================================
# الخلية 9: توليد وعرض الصور الاصطناعية
# ============================================================

# توليد صور جديدة
G.eval()
with torch.no_grad():
    z = torch.randn(12, LATENT_DIM).to(device)
    synthetic_images = G(z).cpu().numpy()

# تحويل من [-1,1] إلى [0,255]
synthetic_images = ((synthetic_images + 1) / 2 * 255).astype(np.uint8)
synthetic_images = synthetic_images.squeeze(1)  # [N, H, W]

# عرض الصور
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for i in range(12):
    row, col = i // 6, i % 6
    axes[row, col].imshow(synthetic_images[i], cmap='gray')
    axes[row, col].set_title(f'Synthetic {i+1}')
    axes[row, col].axis('off')

plt.suptitle('GAN Generated Medical Images', fontsize=14)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'synthetic' / 'gan_samples.png', dpi=150, bbox_inches='tight')
plt.show()

# حفظ الصور الاصطناعية
for i, img in enumerate(synthetic_images):
    np.save(BASE_DIR / 'data' / 'synthetic' / f'synthetic_{i:04d}.npy', img)

# رسم خسائر التدريب
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(g_losses, label='Generator Loss', alpha=0.8)
ax.plot(d_losses, label='Discriminator Loss', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('GAN Training Loss')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / 'data' / 'synthetic' / 'gan_loss.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ تم توليد {len(synthetic_images)} صورة اصطناعية")

# === ملخص المشروع ===
print("\n" + "=" * 50)
print("ملخص المشروع - Medical Image AI Suite")
print("=" * 50)
print(f"  الصور الأصلية:      {len(original_images)}")
print(f"  بعد التعزيز:        {len(all_images)}")
print(f"  التقارير المُعالجة: {len(SAMPLE_REPORTS)}")
print(f"  الفئات المُستخرجة:  {len(all_keys)}")
print(f"  الصور الاصطناعية:   {len(synthetic_images)}")
print(f"  إجمالي الصور:      {len(all_images) + len(synthetic_images)}")
print("=" * 50)